# GRU4Rec — training & validation (Colab)

Trening baseline **GRU4Rec** z **PyTorch Lightning** na danych Yoochoose.

**Logowane metryki (val):**
- `val/loss`, `val/perplexity`
- `val/hit@1`, `val/recall@5`, `val/recall@10`, `val/recall@20`
- `val/mrr@20`, `val/ndcg@20`
- `val/recall@5_pop`, `val/recall@10_pop`, `val/recall@20_pop` (baseline popularności)

Checkpointy: `checkpoints/gru4rec/<run_name>/` na Google Drive.
W&B: `project-nn/adm-project-tgnn` (stałe w `src/config/wandb_settings.py`).

## 0. Konfiguracja użytkownika

In [ ]:
from pathlib import Path

DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/adm-project-tgnn")
REPO_DIR = Path("/content/adm-project-tgnn")
REPO_URL = None  # np. "https://github.com/<org>/adm-project-tgnn.git"

PROCESSED_VARIANT = "subsample_1_32_clicks_only"
RUN_NAME = "gru4rec-baseline"

# Hyperparametry treningu
NUM_EPOCHS = 10
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
EMBEDDING_DIM = 128
HIDDEN_DIM = 128

## 1. Repozytorium i zależności

In [ ]:
import subprocess
import sys

if REPO_URL and not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

if not REPO_DIR.is_dir():
    raise FileNotFoundError(f"Repo not found at {REPO_DIR}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
print(f"Repo ready: {REPO_DIR}")

## 2. Google Drive — mount, walidacja, rozpakowanie danych

In [ ]:
from pprint import pprint

from src.colab.setup import check_drive_layout, prepare_colab_session
from src.config.train_config import TrainConfig

layout = check_drive_layout(DRIVE_PROJECT_DIR)
if not layout.ok:
    raise FileNotFoundError("\n".join(layout.errors))

config = TrainConfig.for_colab(
    DRIVE_PROJECT_DIR,
    model_name="gru4rec",
    processed_variant=PROCESSED_VARIANT,
    run_name=RUN_NAME,
    wandb_run_name=RUN_NAME,
    num_epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
)
config = prepare_colab_session(config)

pprint(
    {
        "processed_dir": config.processed_dir,
        "checkpoint_dir": config.checkpoint_dir,
        "num_epochs": config.num_epochs,
        "batch_size": config.batch_size,
    }
)

## 3. Weights & Biases

In [ ]:
from src.config.wandb_settings import WANDB_ENTITY, WANDB_PROJECT, verify_wandb_access

wandb_info = verify_wandb_access()
print(f"wandb target: {WANDB_ENTITY}/{WANDB_PROJECT}")
for key, value in wandb_info.items():
    print(f"  {key}: {value}")

## 4. Trening GRU4Rec (Lightning)

Uruchamia `train_gru4rec(config)` — pętla treningowa, walidacja z metrykami rankingowymi, checkpointy na Drive i logi do wandb.

In [ ]:
from src.training.train_gru4rec import train_gru4rec

train_gru4rec(config)
print(f"\nTraining finished. Checkpoints: {config.checkpoint_dir}")

## 5. Podsumowanie

Po treningu sprawdź run w [wandb](https://wandb.ai) (`project-nn/adm-project-tgnn`) oraz pliki `epoch_*.ckpt` na Drive.

**Następny krok:** ewaluacja na `test_internal` / `challenge_test` — do dopisania osobno.